# aether — Colab preflight

Первый этап: английское голосовое демо, Google Drive, бюджет 500 юнитов.
Этот notebook готовит среду и отчёт ресурсов. Модель и обучение ещё недоступны.
Откройте в **удалённом управляемом runtime платного Colab**, не в local runtime.
Выделение GPU расходует бюджет даже без обучения: запускайте только вручную.
Для первого BF16 baseline ориентир — GPU с 24+ ГБ, но это не гарантия размещения.
Сначала разрешён только preflight; никаких загрузок весов или optimizer здесь нет.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

CONFIRM_REMOTE_PAID_COLAB = False  # Подтвердите среду в интерфейсе Colab вручную.
RUN_TRAINING = False  # Оставить False; True тоже не разрешает обучение.
BUDGET_UNITS = 500
RUN_ID = ""  # Например english-demo-preflight-001; уникальное имя.
DRIVE_ROOT = "/content/drive/MyDrive/aether"
REPO_URL = "https://github.com/karl4th/aether-v2.git"
GIT_REF = "main"  # Для повторения опыта укажите commit SHA из preflight.json.
UV_VERSION = "0.12.13"

## Проверка выбора среды
Подтверждение ниже — заявление пользователя, не доказательство инфраструктуры.
Наблюдения среды будут записаны отдельно. Обучающий gate остаётся закрытым.


In [ ]:
if RUN_TRAINING:
    raise RuntimeError("Training is unavailable; keep RUN_TRAINING=False")
if not CONFIRM_REMOTE_PAID_COLAB:
    raise RuntimeError("Select a managed remote paid Colab runtime and confirm above")
__import__("google.colab")
if not Path("/content").is_dir() or sys.platform != "linux":
    raise RuntimeError("Expected Colab Linux environment; local runtime is forbidden")
if not RUN_ID or not GIT_REF:
    raise ValueError("Set unique RUN_ID and a Git branch, tag or commit SHA")

## Код из GitHub
Код загружается напрямую из репозитория. `GIT_REF` может быть веткой, тегом или
commit SHA; checkout закрепляется на полученном SHA и записывается в отчёт.
Для приватного репозитория добавьте read-only токен в Colab Secrets с именем
`GITHUB_TOKEN` и разрешите доступ notebook. Токен не вставляется в URL и файлы.
Каждое выполнение setup создаёт отдельный каталог, не меняя предыдущий checkout.


In [ ]:
import base64
import os
import tempfile

from google.colab import userdata


def checkout_source(repo_url, git_ref, workspace, git_env=None):
    if not git_ref or git_ref.startswith("-"):
        raise ValueError("Expected a Git branch, tag or commit SHA")
    project = Path(tempfile.mkdtemp(prefix="aether-src-", dir=workspace))
    env = dict(os.environ if git_env is None else git_env)
    env["GIT_TERMINAL_PROMPT"] = "0"

    def git(*args):
        result = subprocess.run(
            ["git", *args],
            cwd=project,
            env=env,
            check=True,
            capture_output=True,
            text=True,
            timeout=180,
        )
        return result.stdout.strip()

    git("init", "--quiet")
    git("remote", "add", "origin", repo_url)
    git("fetch", "--depth=1", "origin", git_ref)
    revision = git("rev-parse", "FETCH_HEAD^{commit}")
    git("checkout", "--detach", revision)
    for name in ("pyproject.toml", "uv.lock", ".python-version"):
        if not (project / name).is_file():
            raise ValueError("Missing required project file: " + name)
    return project, revision


# Optional read-only GitHub credential; never printed or stored in Git config.
git_env = os.environ.copy()
try:
    token = userdata.get("GITHUB_TOKEN")
except userdata.SecretNotFoundError:
    token = None
if token:
    if REPO_URL != "https://github.com/karl4th/aether-v2.git":
        raise ValueError("Credential use is restricted to the aether repository")
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    git_env.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": "Authorization: Basic " + auth,
        }
    )
    del auth
try:
    PROJECT, SOURCE_REVISION = checkout_source(REPO_URL, GIT_REF, "/content", git_env)
finally:
    git_env.clear()
    del token
print("Source commit:", SOURCE_REVISION)
print("Project:", PROJECT)

## uv и отдельное окружение
В системное ядро устанавливается только зафиксированный uv. Зависимости пакета
управляются `uv sync --locked`; Python пакета — 3.12.14. Модельные зависимости
пока не выбираются и не скачиваются.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==" + UV_VERSION], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(
    UV + ["sync", "--locked", "--no-dev", "--python", "3.12.14"], cwd=PROJECT, check=True
)


def package_run(*arguments):
    return subprocess.run(
        UV + ["run", "--locked", "--no-dev", *arguments],
        cwd=PROJECT,
        check=True,
        capture_output=True,
        text=True,
    )


print(package_run("aether", "--version").stdout)
print(
    package_run("aether", "train", "--config", "configs/model/tiny.json", "--validate-only").stdout
)

## Preflight ресурсов
Процесс пакета читает CPU/RAM/диск и `nvidia-smi`. Не загружает модель, не делает
forward и не проверяет тариф программно. Расход юнитов смотрите в интерфейсе Colab.


In [ ]:
result = package_run(
    "python",
    "-c",
    "import json; from aether.preflight import collect_preflight; "
    "print(json.dumps(collect_preflight()))",
)
report = json.loads(result.stdout)
report.update(
    source_repository=REPO_URL,
    source_ref=GIT_REF,
    source_revision=SOURCE_REVISION,
    uv_version=UV_VERSION,
    budget_units=BUDGET_UNITS,
    language="en",
    scenario="chat_demo",
    user_asserted_remote_paid_colab=CONFIRM_REMOTE_PAID_COLAB,
)
print(json.dumps(report, indent=2, ensure_ascii=False))

## Google Drive и сохранение отчёта
Авторизуйте Drive. Создаётся новый `runs/RUN_ID`; существующий запуск не
перезаписывается. Для повторного preflight задайте новое RUN_ID.


In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Drive is not mounted")
LOCAL_REPORT = PROJECT / "preflight.json"
LOCAL_REPORT.write_text(json.dumps(report, indent=2), encoding="utf-8")
result = package_run(
    "python",
    "-c",
    "from pathlib import Path; import sys; from aether.storage import create_run; "
    "run=create_run(Path(sys.argv[1]),sys.argv[2]); "
    "(run/'preflight.json').write_bytes(Path(sys.argv[3]).read_bytes()); print(run)",
    DRIVE_ROOT,
    RUN_ID,
    str(LOCAL_REPORT),
)
print("Report saved:", result.stdout)
files.download(str(LOCAL_REPORT))

## Артефакты
Выбран кандидат BF16 с начальным кодеком и текстовым токенизатором. Реальная загрузка требует фиксации revision, хешей, архитектуры и проверки совместимости. В этой версии загрузка весов отсутствует.


## Baseline
Пока заблокирован A07/A10–A13/A19. Пришлите preflight.json; модельные операции здесь не подменяются синтетическим демо.


## Pilot / resume / train
Обучение закрыто, RUN_TRAINING=False. Хранилище checkpoint проверено на локальных фикстурах, но восстановление тренера и Colab Drive ещё не проверены. Нужны модель, данные, baseline и все зависимости A22.


## Evaluate / export
Будут доступны после реализации модели и оценки. Этот notebook сохраняет только preflight.json, а не модель или checkpoint.


In [ ]:
if RUN_TRAINING:
    raise RuntimeError("Training is blocked pending verified runtime, model, data and baseline")
print("Preflight complete. Send preflight.json for review; then disconnect the GPU runtime.")